In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:

import numpy as np
import os
PROJECT_PATH = '/content/drive/MyDrive/Git-audio-stego/audio-steganograph/AudioStego/exp'

if os.path.exists(PROJECT_PATH):
    os.chdir(PROJECT_PATH)
    # !pip install -q librosa soundfile tensorflow scikit-learn seaborn
else:
    print(f" Không tìm thấy thư mục tại {PROJECT_PATH}")



In [ ]:
import os
import glob
import random
import shutil
from PIL import Image

TIMIT_DIR = "/content/drive/MyDrive/HK2-20252026/data/timit"
VOC_DIR = "/content/drive/MyDrive/HK2-20252026/data/pascal-voc-2012/1/VOC2012/JPEGImages"

NUM_AUDIO_NEEDED = 1000
NUM_IMAGE_NEEDED = 500
BASE_OUTPUT_DIR = "/content/drive/MyDrive/HK2-20252026/data_clean"
AUDIO_OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, "timit_1000")
IMAGE_OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, "voc_500")

os.makedirs(AUDIO_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMAGE_OUTPUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)


def get_files_recursive(directory, extensions):
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(directory, "**", f"*.{ext}"), recursive=True))
    return files

def is_riff_wav(filepath):
    """True nếu file có header RIFF/RIFX/RF64 (đọc được bởi scipy/soundfile).
    TIMIT gốc dùng NIST SPHERE header -> không đọc được -> phải loại.
    """
    try:
        with open(filepath, 'rb') as f:
            header = f.read(4)
        return header in (b'RIFF', b'RIFX', b'RF64')
    except Exception:
        return False


def filter_valid_audio(directory, num_needed, log_fn=print):
    log_fn(f"[*] Quét toàn bộ .wav/.WAV trong {directory} ...")
    all_candidates = get_files_recursive(directory, ["wav", "WAV"])
    log_fn(f"[*] Tổng số file tìm thấy (chưa lọc header): {len(all_candidates)}")

    random.shuffle(all_candidates)

    valid_files = []
    nist_count = 0
    other_invalid_count = 0
    checked = 0

    for f in all_candidates:
        checked += 1
        if is_riff_wav(f):
            valid_files.append(f)
        else:
            with open(f, 'rb') as fh:
                header = fh.read(4)
            if header == b'NIST':
                nist_count += 1
            else:
                other_invalid_count += 1

        if len(valid_files) >= num_needed:
            break

    log_fn(f"[*] Đã kiểm tra {checked}/{len(all_candidates)} file để đủ {len(valid_files)} file hợp lệ")
    log_fn(f"[*] Kết quả lọc: {len(valid_files)} RIFF hợp lệ | "
           f"{nist_count} NIST SPHERE (loại) | {other_invalid_count} khác/lỗi (loại)")

    if len(valid_files) < num_needed:
        log_fn(f"[WARN] Chỉ tìm được {len(valid_files)}/{num_needed} file hợp lệ trong toàn bộ pool!")

    return valid_files[:num_needed]

def filter_valid_images(directory, num_needed, log_fn=print):
    log_fn(f"[*] Quét toàn bộ .jpg/.jpeg trong {directory} ...")
    all_candidates = get_files_recursive(directory, ["jpg", "JPG", "jpeg", "JPEG"])
    log_fn(f"[*] Tổng số ảnh tìm thấy: {len(all_candidates)}")

    random.shuffle(all_candidates)

    valid_files = []
    corrupt_count = 0
    checked = 0

    for f in all_candidates:
        checked += 1
        try:
            with Image.open(f) as img:
                img.verify()  # chỉ check header/cấu trúc, không decode toàn bộ pixel -> nhanh
            valid_files.append(f)
        except Exception:
            corrupt_count += 1

        if len(valid_files) >= num_needed:
            break

    log_fn(f"[*] Đã kiểm tra {checked}/{len(all_candidates)} ảnh để đủ {len(valid_files)} ảnh hợp lệ")
    log_fn(f"[*] Kết quả lọc: {len(valid_files)} ảnh hợp lệ | {corrupt_count} ảnh lỗi/corrupt (loại)")

    if len(valid_files) < num_needed:
        log_fn(f"[WARN] Chỉ tìm được {len(valid_files)}/{num_needed} ảnh hợp lệ trong toàn bộ pool!")

    return valid_files[:num_needed]

def copy_files(file_list, dest_dir, log_fn=print):
    """Copy từng file vào dest_dir (flat, không giữ cấu trúc thư mục con).
    Nếu trùng tên (VD nhiều speaker đều có SA1.WAV), tự thêm số thứ tự để tránh đè file.
    """
    log_fn(f"[*] Đang copy {len(file_list)} file vào {dest_dir} ...")
    used_names = set()
    copied_paths = []

    for src in file_list:
        base = os.path.basename(src)
        name, ext = os.path.splitext(base)
        dest_name = base
        counter = 1
        while dest_name in used_names:
            dest_name = f"{name}_{counter}{ext}"
            counter += 1
        used_names.add(dest_name)

        dest_path = os.path.join(dest_dir, dest_name)
        shutil.copy2(src, dest_path)
        copied_paths.append(dest_path)

    log_fn(f"[*] Copy xong: {len(copied_paths)} file tại {dest_dir}")
    return copied_paths

print("=" * 60)
print("BƯỚC 1: Lọc audio TIMIT hợp lệ")
print("=" * 60)
valid_audio = filter_valid_audio(TIMIT_DIR, NUM_AUDIO_NEEDED)

print()
print("=" * 60)
print("BƯỚC 2: Copy audio vào thư mục đích")
print("=" * 60)
copied_audio = copy_files(valid_audio, AUDIO_OUTPUT_DIR)

print()
print("=" * 60)
print("BƯỚC 3: Lọc ảnh VOC hợp lệ")
print("=" * 60)
valid_images = filter_valid_images(VOC_DIR, NUM_IMAGE_NEEDED)

print()
print("=" * 60)
print("BƯỚC 4: Copy ảnh vào thư mục đích")
print("=" * 60)
copied_images = copy_files(valid_images, IMAGE_OUTPUT_DIR)

with open(os.path.join(BASE_OUTPUT_DIR, "audio_list.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(copied_audio))
with open(os.path.join(BASE_OUTPUT_DIR, "image_list.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(copied_images))

print()
print("=" * 60)
print("HOÀN TẤT")
print("=" * 60)
print(f"Audio sạch (RIFF, n={len(copied_audio)}) -> {AUDIO_OUTPUT_DIR}")
print(f"Ảnh sạch   (n={len(copied_images)})       -> {IMAGE_OUTPUT_DIR}")
print(f"Danh sách path gốc lưu tại: {BASE_OUTPUT_DIR}/audio_list.txt, image_list.txt")
print()
print("Từ giờ, mọi case (1-7) trong run_case_limited.py chỉ cần dùng:")
print(f"  --input_dir  {AUDIO_OUTPUT_DIR}")
print(f"  --secret_dir {IMAGE_OUTPUT_DIR}")
print("không cần --max_files/--limit/random nữa, vì cả 2 thư mục đã đúng sẵn 1000/500 file sạch.")

BƯỚC 1: Lọc audio TIMIT hợp lệ
[*] Quét toàn bộ .wav/.WAV trong /content/drive/MyDrive/HK2-20252026/data/timit ...
[*] Tổng số file tìm thấy (chưa lọc header): 12600
[*] Đã kiểm tra 1978/12600 file để đủ 1000 file hợp lệ
[*] Kết quả lọc: 1000 RIFF hợp lệ | 978 NIST SPHERE (loại) | 0 khác/lỗi (loại)

BƯỚC 2: Copy audio vào thư mục đích
[*] Đang copy 1000 file vào /content/drive/MyDrive/HK2-20252026/data_clean/timit_1000 ...
[*] Copy xong: 1000 file tại /content/drive/MyDrive/HK2-20252026/data_clean/timit_1000

BƯỚC 3: Lọc ảnh VOC hợp lệ
[*] Quét toàn bộ .jpg/.jpeg trong /content/drive/MyDrive/HK2-20252026/data/pascal-voc-2012/1/VOC2012/JPEGImages ...
[*] Tổng số ảnh tìm thấy: 17125
[*] Đã kiểm tra 500/17125 ảnh để đủ 500 ảnh hợp lệ
[*] Kết quả lọc: 500 ảnh hợp lệ | 0 ảnh lỗi/corrupt (loại)

BƯỚC 4: Copy ảnh vào thư mục đích
[*] Đang copy 500 file vào /content/drive/MyDrive/HK2-20252026/data_clean/voc_500 ...
[*] Copy xong: 500 file tại /content/drive/MyDrive/HK2-20252026/data_clean/voc_

In [ ]:
!python run_case_limited.py \
  --case_id 5 \
  --input_dir  "/content/drive/MyDrive/HK2-20252026/data_clean/timit_1000" \
  --secret_dir "/content/drive/MyDrive/HK2-20252026/data_clean/voc_500" \
  --output_base "/content/drive/MyDrive/HK1-20252026/Steganography/RUN-TIMIT/Timit_voc_500_128_20" \
  --limit 500 \
  --size 128

[*] Scanning Cover: /content/drive/MyDrive/HK2-20252026/data_clean/timit_1000...
[*] Scanning Secret: /content/drive/MyDrive/HK2-20252026/data_clean/voc_500...
--------------------------------------------------
[*] Mode: DATASET GENERATION (VN Time)
[*] Cover: 1000 | Secret Pool: 500
[*] Image Size Target: 128x128
[*] Capacity-aware matching: ON
--------------------------------------------------
[*] Đang tính capacity cho từng audio cover...
[*] Đang đo dung lượng từng ảnh secret (size file gốc, dùng làm proxy)...
[*] Capacity-aware matching: 1000 pairs | Unmatched (no image fits, dùng ảnh nhỏ nhất làm fallback): 5
[1000/1000] SA1.WAV_46.wav -> OK (PSNR:98.9dB | k=1 | CPU:46.9% | RAM:56.8MB)
[DONE] Finished.
Structure: /content/drive/MyDrive/HK1-20252026/Steganography/RUN-TIMIT/Timit_voc_500_128_20/5_Random_Adaptive_ContentSalt_20260621_013041/[logs, cover, stego]
Total: 1000 | Created: 1000 | Skipped: 0


In [ ]:
!python run_case_limited.py \
  --case_id 5 \
  --input_dir  "/content/drive/MyDrive/HK2-20252026/data_clean/timit_1000" \
  --secret_dir "/content/drive/MyDrive/HK2-20252026/data_clean/voc_500" \
  --output_base "/content/drive/MyDrive/HK1-20252026/Steganography/RUN-TIMIT/Timit_voc_500_256_20" \
  --limit 500 \
  --size 256

[*] Scanning Cover: /content/drive/MyDrive/HK2-20252026/data_clean/timit_1000...
[*] Scanning Secret: /content/drive/MyDrive/HK2-20252026/data_clean/voc_500...
--------------------------------------------------
[*] Mode: DATASET GENERATION (VN Time)
[*] Cover: 1000 | Secret Pool: 500
[*] Image Size Target: 256x256
[*] Capacity-aware matching: ON
--------------------------------------------------
[*] Đang tính capacity cho từng audio cover...
[*] Đang đo dung lượng từng ảnh secret (size file gốc, dùng làm proxy)...
[*] Capacity-aware matching: 1000 pairs | Unmatched (no image fits, dùng ảnh nhỏ nhất làm fallback): 5
[1000/1000] SX392.WAV_1.wav -> OK (PSNR:94.0dB | k=1 | CPU:52.9% | RAM:56.8MB)
[DONE] Finished.
Structure: /content/drive/MyDrive/HK1-20252026/Steganography/RUN-TIMIT/Timit_voc_500_256_20/5_Random_Adaptive_ContentSalt_20260621_014259/[logs, cover, stego]
Total: 1000 | Created: 1000 | Skipped: 0


In [ ]:
!python run_case_limited.py \
  --case_id 5 \
  --input_dir  "/content/drive/MyDrive/HK2-20252026/data_clean/timit_1000" \
  --secret_dir "/content/drive/MyDrive/HK2-20252026/data_clean/voc_500" \
  --output_base "/content/drive/MyDrive/HK1-20252026/Steganography/RUN-TIMIT/Timit_voc_500_512_20" \
  --limit 500 \
  --size 512

[*] Scanning Cover: /content/drive/MyDrive/HK2-20252026/data_clean/timit_1000...
[*] Scanning Secret: /content/drive/MyDrive/HK2-20252026/data_clean/voc_500...
--------------------------------------------------
[*] Mode: DATASET GENERATION (VN Time)
[*] Cover: 1000 | Secret Pool: 500
[*] Image Size Target: 512x512
[*] Capacity-aware matching: ON
--------------------------------------------------
[*] Đang tính capacity cho từng audio cover...
[*] Đang đo dung lượng từng ảnh secret (size file gốc, dùng làm proxy)...
[*] Capacity-aware matching: 1000 pairs | Unmatched (no image fits, dùng ảnh nhỏ nhất làm fallback): 5
[1000/1000] SX417.WAV_1.wav -> OK (PSNR:88.6dB | k=2 | CPU:33.2% | RAM:56.7MB)
[DONE] Finished.
Structure: /content/drive/MyDrive/HK1-20252026/Steganography/RUN-TIMIT/Timit_voc_500_512_20/5_Random_Adaptive_ContentSalt_20260621_011044/[logs, cover, stego]
Total: 1000 | Created: 1000 | Skipped: 0
